# Prompt evaluation

Requirements:
- Use preferrably Python `3.12.13` venvs.
    - Using older version might not stream events chunk by chunk, instead it might wait for the **entire** response before printing it.
- Add a `.env` file in the same folder as this notebook with a fields:
    - ANTHROPIC_BASE_URL = `<url_here>`
    - ANTHROPIC_AUTH_TOKEN = "`<jwt_token_here>`"

## ATENTION

- Prefer "claude-4-5-haiku" for this notebook.

## 0. Setup

### 0.1. Env and Client

In [1]:
# Load env variables and create client

## 1 Env
from dotenv import load_dotenv

load_dotenv()

## 2 Client
from anthropic import Anthropic

client = Anthropic()
model = "bedrock/anthropic.claude-4-5-haiku"

### 0.2. Helper functions

In [2]:
# Helper functions
def add_user_message(
    messages, 
    text,
):
    user_message = {
        "role": "user", 
        "content": text,
    }
    
    messages.append(user_message)


def add_assistant_message(
    messages, 
    text,
):
    assistant_message = {
        "role": "assistant", 
        "content": text,
    }

    messages.append(assistant_message)


def chat(
    messages, 
    system=None, 
    temperature=1.0, 
    stop_sequences=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system
    
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

## 1. Prompt Eval

In [3]:
## Dataset generation helper

import json

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []

    add_user_message(
        messages, 
        prompt,
    )

    prefill = "```json"
    add_assistant_message(
        messages,
        prefill,
    )

    stop_sequences = ["```"]
    text = chat(
        messages,
        stop_sequences=stop_sequences,
    )

    return json.loads(text)

In [4]:
## Generate dataset and print it

# dataset = generate_dataset()
# print(dataset)

In [5]:
## Save dataset to a file

# with open('dataset.json', 'w') as f:
#     json.dump(
#         dataset, 
#         f, 
#         indent=2,
#     )

# print("Dataset saved to dataset.json")

## 2. Running the eval

In [6]:
# Helpers

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [7]:
# Read and run eval pipeline

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [8]:
# Print results

print(json.dumps(results, indent=1))

[
 {
  "output": "# AWS S3 Bucket Region Extractor\n\nHere's a Python function that extracts the AWS region from an S3 bucket ARN:\n\n```python\ndef extract_s3_region(arn: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket ARN.\n    \n    S3 bucket ARNs have the format: arn:aws:s3:::bucket-name\n    Note: S3 bucket ARNs don't contain region information in the standard format.\n    However, this function handles various ARN formats and returns the region if present.\n    \n    Args:\n        arn (str): The S3 bucket ARN\n        \n    Returns:\n        str: The AWS region, or 'us-east-1' as default\n        \n    Examples:\n        >>> extract_s3_region('arn:aws:s3:::my-bucket')\n        'us-east-1'\n        >>> extract_s3_region('arn:aws:s3:us-west-2::my-bucket')\n        'us-west-2'\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        return 'us-east-1'\n    \n    # Split the ARN by colons\n    parts = arn.split(':')\n    \n    # Standard S3 bucket AR

## 3. Model based grader

### 3.1. Helpers

In [9]:
# New grader function

def grade_by_model(
    test_case,
    output,
): 
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []

    prefill_sequence = "```json"
    stop_sequence = "```"

    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, prefill_sequence)

    eval_text = chat(
        messages,
        temperature=0,
        stop_sequences=[stop_sequence],
    )

    return json.loads(eval_text)

In [10]:
# Eval helpers override

from statistics import mean

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grader
    model_grade = grade_by_model(
        test_case,
        output,
    )

    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    strengths = model_grade["strengths"]
    weaknesses = model_grade["weaknesses"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
        "strengths": strengths,
        "weaknesses": weaknesses,
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    # Average grader score
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score:.2f}")

    return results

### 3.2. Read dataset

In [11]:
# Open dataset.json

with open("dataset.json", "r") as f:
    dataset = json.load(f)

### 3.3. Run eval

In [12]:
# Run eval pipeline

results = run_eval(dataset)

Average score: 6.00


In [13]:
# Print results

print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket ARN Region Extractor\n\nHere's a comprehensive solution:\n\n```python\nimport re\nfrom typing import Optional\n\ndef extract_region_from_s3_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket ARN.\n    \n    S3 bucket ARNs can have the following formats:\n    - arn:aws:s3:::bucket-name (no region specified, defaults to us-east-1)\n    - arn:aws:s3:region:account-id:bucket/bucket-name (regional format)\n    \n    Args:\n        arn: The S3 bucket ARN string\n        \n    Returns:\n        The AWS region, or 'us-east-1' as default if not specified\n        \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    default_region = 'us-east-1'\n    \n    # Validate ARN format\n    if not arn or not isinstance(arn, str):\n        raise ValueError(\"ARN must be a non-empty string\")\n    \n    # Split the ARN by colons\n    parts = arn.split(':')\n    \n    # Standard ARN format: arn:partition:service:r